
# 지자체 혼인 감시 백테스트 (KOSIS 시군구 월별, 1997~2025)

**질문:** 지자체에 "우리 시 혼인 건수가 예측을 벗어나면 알려드립니다"라는 감시 서비스를 판다면, 과거에 이 경보는 실제 이탈을 **몇 개월 만에, 얼마나 정확히** 잡았을까?

**현실 제약 (2026-10 KOSIS 확인)**

| 자료 | 단위 | 공표 |
|---|---|---|
| `DT_1B83A35` 시도/시군구/월별 혼인 | 시군구 × 월 | **다음 해 초에 1년치를 한 번에** (지금 2025.12까지) |
| `DT_1B8000G` 월별 인구동향 | 시도 × 월 | **매달, 약 2개월 지연** (지금 2026.07까지) |

그래서 두 가지를 따로 잽니다.
1. **시군구 월별 경보의 성능:** 시군구 월별 자료를 매달 받을 수 있다고 가정합니다. 실제로는 지자체 자체 혼인신고 자료가 있어야 실시간으로 가능합니다.
2. **실시간으로 지금 가능한 것:** 시도 월간 발표만으로 소속 시군구의 이탈을 확정치 공표보다 얼마나 먼저 알 수 있는지.

**설계 (채점 기간 결과를 보기 전에 고정)**
- **예측 (플랜 A, 모든 지역에 같은 방식):** 작년 연간 건수 × 최근 5년 평균 월별 비중
- **이탈 해:** 연간 실제/예측이 ±10% 이상. **정상 해:** ±5% 미만. 그 사이는 판단에서 제외합니다.
- **평가 대상:** 직전 연도 혼인 300건 이상인 시·군·자치구. 일반구와 행정구역 개편 전 계열은 제외합니다.
- **경보:**
  - 월별 잔차(실제 − 예측)를 과분산으로 표준화한 양방향 CUSUM입니다. 허용 폭 0.25이고, 해마다 1월에 새로 시작합니다.
  - 기준값은 **보정 기간(2003~2012)에서 정상 해의 잘못된 경보율이 10% 이하가 되는 가장 민감한 값**으로 고르고, **채점 기간(2013~2025)**으로 평가합니다.
- **비교:**
  - 단순 규칙(1월부터 누적 이탈률이 ±x%를 넘으면 경보)
  - 시도 대비 경보: 같은 달 시도 실제 × 작년 시도 내 비중. 전국·시도 흐름을 걸러낸 **지역 고유** 이탈입니다.

자료: 통계청 KOSIS `DT_1B83A35` (가공). 노트북은 저장소의 CSV를 쓰므로 키가 필요 없습니다. 맨 아래 셀로 최신 자료를 다시 받을 수 있습니다.


In [ ]:
import os, numpy as np, pandas as pd

DATA = "data_kosis/marriages_monthly_sigungu_1997_2025.csv"
DATA_URL = ("https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/"
            "experiments/data_kosis/marriages_monthly_sigungu_1997_2025.csv")

# ---- 미리 정한 것 (채점 기간 결과를 보기 전에 고정) ----
CAL = range(2003, 2013)      # 경보 기준을 정하는 기간 (예측 연도)
TEST = range(2013, 2026)     # 채점 기간
MIN_SIZE = 300               # 직전 연도 혼인 300건 이상 시군구만 평가
DEV, CLEAR = 0.10, 0.05      # 연간 실제/예측 −1 이 ±10% 이상 = 이탈 해, ±5% 미만 = 정상 해 (사이는 제외)
K = 0.25                     # CUSUM 허용 폭 (표준화 단위)
MAX_FA = 0.10                # 보정 기간 정상 해의 잘못된 경보율 10% 이하가 되는 가장 민감한 기준을 고름
LAG = 2                      # 시도 월별 인구동향 공표 지연 (개월)
RELEASE_M = 15               # 시군구 월별 확정치 공표 시점: 다음 해 3월쯤 (예측 연도 1월 = 1)

def load(path=DATA):
    d = pd.read_csv(path if os.path.exists(path) else DATA_URL, dtype={"code": str, "ym": str})
    d["year"] = d.ym.str[:4].astype(int); d["month"] = d.ym.str[4:].astype(int)
    return d

def wide(d, codes):
    return d[d.code.isin(codes)].pivot_table(index="code", columns=["year", "month"], values="marriages")

def forecasts(W):
    """플랜 A 예측: 작년 연간 건수 × 최근 5년 평균 월별 비중. {(지역, 연도): (월별 예측, 월별 실제, 작년 연간)}"""
    out = {}
    years = sorted({y for y, _ in W.columns})
    for code, row in W.iterrows():
        for Y in years:
            need = [Y - i for i in range(6)]
            if any(y not in years for y in need): continue
            M = np.array([[row.get((y, m), np.nan) for m in range(1, 13)] for y in need])
            if np.isnan(M).any(): continue
            shares = (M[1:] / M[1:].sum(1, keepdims=True)).mean(0)
            out[(code, Y)] = (M[1].sum() * shares, M[0], M[1].sum())
    return out

def relative_forecasts(W, S):
    """시도 대비 예측: 이번 달 시도 실제 × 작년 이 시군구의 시도 내 비중. 전국·시도 흐름을 걸러낸 '지역 고유' 이탈용."""
    out = {}
    years = sorted({y for y, _ in W.columns})
    for code, row in W.iterrows():
        if code[:2] not in S.index: continue
        srow = S.loc[code[:2]]
        for Y in years:
            if Y - 1 not in years: continue
            g = lambda r, y: np.array([r.get((y, m), np.nan) for m in range(1, 13)])
            a, ap, s, sp = g(row, Y), g(row, Y - 1), g(srow, Y), g(srow, Y - 1)
            if np.isnan(np.r_[a, ap, s, sp]).any(): continue
            out[(code, Y)] = (s * ap.sum() / sp.sum(), a, ap.sum())
    return out

def phi_from(F, years, min_size=MIN_SIZE):
    """과분산: 분산이 평균의 몇 배인지 (보정 기간에서만 추정)."""
    return float(np.mean([((a - mu) ** 2 / mu).mean() for (c, y), (mu, a, A) in F.items() if y in years and A >= min_size]))

def cusum_month(mu, actual, phi, h, k=K):
    """1월부터 양방향 CUSUM. 첫 경보 월(1~12)과 방향(+1/−1). 없으면 (None, 0)."""
    z = (actual - mu) / np.sqrt(phi * mu)
    cp = cm = 0.0
    for m in range(12):
        cp = max(0.0, cp + z[m] - k); cm = max(0.0, cm - z[m] - k)
        if cp > h: return m + 1, 1
        if cm > h: return m + 1, -1
    return None, 0

def ytd_month(mu, actual, x):
    """단순 규칙: 1월부터 누적 실제 / 누적 예측 − 1 이 ±x를 넘는 첫 달."""
    r = np.cumsum(actual) / np.cumsum(mu) - 1
    for m in range(12):
        if r[m] > x: return m + 1, 1
        if r[m] < -x: return m + 1, -1
    return None, 0

def evaluate(F, years, rule, min_size=MIN_SIZE):
    rows = []
    for (c, y), (mu, a, A) in F.items():
        if y not in years or A < min_size: continue
        m, s = rule(mu, a)
        rows.append(dict(code=c, year=y, dev=a.sum() / mu.sum() - 1, alarm_m=m, alarm_dir=s))
    R = pd.DataFrame(rows)
    pos, neg = R[R.dev.abs() >= DEV], R[R.dev.abs() < CLEAR]
    hit = pos[pos.alarm_dir == np.sign(pos.dev)]
    return R, dict(지역연도=len(R), 이탈해=len(pos), 정상해=len(neg),
                   감지율=round(len(hit) / max(len(pos), 1), 3),
                   상반기내_감지율=round(float((hit.alarm_m <= 6).sum()) / max(len(pos), 1), 3),
                   감지월_중앙값=float(hit.alarm_m.median()) if len(hit) else np.nan,
                   잘못된경보율=round(float((neg.alarm_dir != 0).mean()), 3) if len(neg) else np.nan)

def pick(F, make_rule, grid, min_size=MIN_SIZE):
    """보정 기간에서 잘못된 경보율 ≤ MAX_FA 인 가장 민감한 기준."""
    for g in grid:
        _, s = evaluate(F, CAL, make_rule(g), min_size)
        if s["잘못된경보율"] <= MAX_FA:
            return g, s
    return g, s

def sido_lead(F, FS, phi_s, h_s):
    """시군구 이탈 해 중 같은 해 소속 시도의 월간 경보(같은 방향)로 미리 안 비율과, 시군구 확정치보다 앞선 개월."""
    rows = []
    for (c, y), (mu, a, A) in F.items():
        if y not in TEST or A < MIN_SIZE: continue
        r = a.sum() / mu.sum() - 1
        if abs(r) < DEV: continue
        smu, sa, _ = FS.get((c[:2], y), (None, None, None))
        if smu is None: continue
        m, s = cusum_month(smu, sa, phi_s, h_s)
        ok = s == np.sign(r)
        rows.append(dict(code=c, year=y, dev=r, sido_alarm_m=m if ok else np.nan,
                         lead=RELEASE_M - (m + LAG) if ok else np.nan))
    return pd.DataFrame(rows)


## 1. 경보 성능 (채점 기간 2013~2025)

In [ ]:
d = load()
name = d.drop_duplicates("code").set_index("code").name
sgg = [c for c in d.code.unique() if len(c) == 5 and c.endswith("0") and "변동전" not in name[c]]   # 시·군·자치구 (일반구·개편 전 계열 제외)
sido = [c for c in d.code.unique() if len(c) == 2 and c not in ("00", "90")]
W, S = wide(d, sgg), wide(d, sido)
F, FR, FS = forecasts(W), relative_forecasts(W, S), forecasts(S)
phi, phi_r, phi_s = phi_from(F, CAL), phi_from(FR, CAL), phi_from(FS, CAL, 0)

H_GRID = [2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 10, 12]
h_abs, cal_abs = pick(F, lambda h: (lambda mu, a: cusum_month(mu, a, phi, h)), H_GRID)
x_ytd, cal_ytd = pick(F, lambda x: (lambda mu, a: ytd_month(mu, a, x)), [0.05, 0.075, 0.1, 0.125, 0.15, 0.2, 0.25, 0.3, 0.4])
h_rel, cal_rel = pick(FR, lambda h: (lambda mu, a: cusum_month(mu, a, phi_r, h)), H_GRID)
h_s, cal_s = pick(FS, lambda h: (lambda mu, a: cusum_month(mu, a, phi_s, h)), H_GRID, min_size=0)

R_abs, t_abs = evaluate(F, TEST, lambda mu, a: cusum_month(mu, a, phi, h_abs))
_, t_ytd = evaluate(F, TEST, lambda mu, a: ytd_month(mu, a, x_ytd))
R_rel, t_rel = evaluate(FR, TEST, lambda mu, a: cusum_month(mu, a, phi_r, h_rel))
_, t_s = evaluate(FS, TEST, lambda mu, a: cusum_month(mu, a, phi_s, h_s), min_size=0)

print(f"시군구 {len(sgg)}곳 · 시도 {len(sido)}곳 · 과분산 φ: 절대 {phi:.2f}, 시도 대비 {phi_r:.2f}, 시도 {phi_s:.1f}")
summary = pd.DataFrame({
    f"시군구 절대 CUSUM (h={h_abs})": t_abs,
    f"시군구 누적 이탈 규칙 (±{x_ytd:.0%})": t_ytd,
    f"시군구 시도 대비 CUSUM (h={h_rel})": t_rel,
    f"시도 절대 CUSUM (h={h_s})": t_s,
}).T
summary


In [ ]:

# 규모별: 큰 도시는 작은 차이도 통계적으로 유의해져 경보가 잦아짐
R = R_abs.copy()
R["작년건수"] = [F[(c, y)][2] for c, y in zip(R.code, R.year)]
R["규모"] = pd.cut(R.작년건수, [0, 1000, 3000, 1e9], labels=["300~1천", "1천~3천", "3천 이상"])
R["이탈"] = R.dev.abs() >= DEV; R["정상"] = R.dev.abs() < CLEAR
R["감지"] = R.이탈 & (R.alarm_dir == np.sign(R.dev))
R.groupby("규모", observed=True).apply(lambda x: pd.Series(dict(
    지역연도=len(x), 이탈해=int(x.이탈.sum()), 감지율=round(x.감지.sum() / max(x.이탈.sum(), 1), 2),
    정상해=int(x.정상.sum()), 잘못된경보율=round(float((x[x.정상].alarm_dir != 0).mean()), 2))))


## 2. 지금 실시간으로 가능한 것: 시도 월간 경보로 시군구 이탈을 미리 알기

In [ ]:

L = sido_lead(F, FS, phi_s, h_s)
print(f"시군구 이탈 해 {len(L)}건 중 같은 해 소속 시도 경보(같은 방향)로 미리 안 비율: {L.lead.notna().mean():.0%}")
print(f"미리 안 경우, 시군구 확정치 공표(다음 해 3월)보다 앞선 개월 중앙값: {L.lead.median():.0f}개월 (시도 공표 지연 {LAG}개월 반영)")
L.groupby("year").apply(lambda g: pd.Series(dict(이탈해=len(g), 미리앎=round(g.lead.notna().mean(), 2),
                                                 앞선개월_중앙값=g.lead.median()))).T


## 3. 그림

In [ ]:

import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"); plt.rcParams["font.family"] = "NanumGothic"
except Exception:
    pass

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
# (가) 예시: 수원시 2024
mu, a, _ = F[("31010", 2024)]
ax[0].plot(range(1, 13), mu, "o--", label="예측 (작년 × 계절)"); ax[0].plot(range(1, 13), a, "o-", label="실제")
m, s = cusum_month(mu, a, phi, h_abs)
if m: ax[0].axvline(m, color="r", ls=":", label=f"경보 {m}월")
ax[0].set_title("수원시 2024: 월별 혼인"); ax[0].set_xlabel("월"); ax[0].legend(fontsize=8)
# (나) 연도별 이탈 해와 감지
Y = R_abs.assign(이탈=R_abs.dev.abs() >= DEV, 감지=(R_abs.dev.abs() >= DEV) & (R_abs.alarm_dir == np.sign(R_abs.dev))).groupby("year")[["이탈", "감지"]].sum()
Y.plot.bar(ax=ax[1]); ax[1].set_title("연도별 시군구 이탈 해와 경보로 잡은 수"); ax[1].set_xlabel("")
# (다) 감지 월 분포
hit = R_abs[(R_abs.dev.abs() >= DEV) & (R_abs.alarm_dir == np.sign(R_abs.dev))]
ax[2].hist(hit.alarm_m, bins=np.arange(0.5, 13.5), rwidth=0.85)
ax[2].set_title("이탈 해를 처음 경보한 달"); ax[2].set_xlabel("월")
plt.tight_layout(); plt.show()



## 4. 사전 실행 결과와 해석

**경보 성능 (채점 기간 2013~2025, 기준값은 2003~2012에서만 정함)**

| 경보 | 기준 | 이탈 해 | 감지율 | 상반기 내 | 감지 월 중앙값 | 잘못된 경보율 |
|---|---|---|---|---|---|---|
| **시군구 절대 CUSUM** | h=4.5 | 610 | **80%** | 32% | 7월 | **5.6%** |
| 시군구 누적 이탈 단순 규칙 | ±25% | 610 | 27% | 25% | 1월 | 7.7% |
| **시군구 시도 대비 CUSUM** (지역 고유 이탈) | h=4.5 | 251 | **90%** | 32% | 8월 | 6.0% |
| 시도 절대 CUSUM | h=7 | 41 | 59% | 15% | 9.5월 | 2.0% |

**규모별 (시군구 절대 CUSUM)**

| 직전 연도 혼인 | 이탈 해 | 감지율 | 잘못된 경보율 |
|---|---|---|---|
| 300~1천 건 | 277 | 57% | 1% |
| 1천~3천 건 | 290 | 99% | 4% |
| **3천 건 이상 (수원·용인·성남 등)** | 43 | 100% | **27%** |

**시도 월간 경보로 미리 알기 (지금 실시간으로 가능)**
- 시군구 이탈 해 610건 중 **51%**는 같은 해 소속 시도의 월간 경보(같은 방향)로 미리 알 수 있었습니다.
- 그 경우 시군구 확정치 공표보다 **6개월(중앙값)** 앞섰습니다.
- 전국적으로 크게 움직인 해에 특히 잘 잡았습니다. 2020년 감소 때 80%, 2024년 반등 때 85%입니다.
- 지역 고유 이탈이 많았던 2015·2018·2022·2023년에는 0%였습니다.

### 알게 된 것
1. **월별 감시는 작동합니다.** 연간 ±10% 이탈의 80%를 잡고, 잘못된 경보는 정상 해의 6%입니다. 보정 기간에 정한 기준값을 그대로 썼는데도 채점 기간에 성능이 유지됐습니다.
2. **단순 누적 규칙은 크게 못합니다(27%).** 연초 한두 달의 잡음에 반응하기 때문입니다. 여기서는 "단순한 것"이 답이 아닙니다. 월별 잡음을 감안하는 CUSUM이 필요합니다.
3. **이탈의 대부분은 전국·시도 흐름입니다.** 시도 대비로 걸러내면 지역 고유 이탈은 610건에서 251건으로 줄어듭니다. 지자체에 더 쓸모 있는 것은 이 **지역 고유 경보**이고, 감지율도 90%로 더 높습니다.
4. **실시간 시군구 경보는 공개 자료만으로는 불가능합니다.**
   - 지금 가능한 것은 시도 월간 경보뿐이고, 이것으로는 전국·시도 흐름에 따른 이탈의 절반 정도를 6개월 먼저 압니다.
   - **지역 고유 이탈을 실시간으로 잡으려면 지자체가 자체 혼인신고 월별 자료를 줘야 합니다.** 고객이 데이터 공급자이자 휘슬이 되는 구조입니다.
5. **영업 대상인 큰 도시에서는 경보가 너무 잦습니다(27%).** 건수가 많으면 2~3% 차이도 통계적으로 유의해지기 때문입니다. 큰 도시는 "통계적으로 유의 + 누적 이탈 ±5% 이상"처럼 **실무적 크기 조건**을 함께 걸어야 합니다. 이 조건은 이번 결과를 본 뒤 떠올린 것이라 다음 백테스트에서 미리 정하고 다시 채점해야 합니다.

### 영업 문구에 쓸 수 있는 숫자 (출처·조건과 함께)
- "2013~2025년, 연 300건 이상 시군구의 연간 ±10% 혼인 이탈 610건 중 80%를 월별 감시로 잡았습니다(잘못된 경보는 정상 해의 6%)."
- "공개된 시도 월간 통계만으로도 이 중 절반은 시군구 확정 통계보다 평균(중앙값) 6개월 먼저 알 수 있었습니다."

### 한계
- 시군구 월별 경보 성능은 "월별 자료를 매달 받는다"는 가정에서 나온 것입니다.
- 시도 단위는 확정치(`DT_1B83A35`)로 계산했습니다. 실제 월간 발표는 잠정치(`DT_1B8000G`)라 값이 조금 다를 수 있습니다.
- 확정치 공표 시점(다음 해 3월)은 근사값입니다.
- 보정 기간의 시도 이탈 해가 4건뿐이라, 시도 기준값(h=7)은 근거가 약합니다.
- 행정구역 개편(창원·청주 통합 등) 이전 계열은 제외했습니다.


## 부록: 최신 자료 다시 받기 (KOSIS 키 필요)
코랩 왼쪽 열쇠 아이콘(보안 비밀)에 `KOSIS_API_KEY`를 넣고 아래 셀에서 `REFRESH = True`로 바꿔 실행합니다.

In [ ]:

REFRESH = False
if REFRESH:
    import json, time, urllib.parse, urllib.request
    try:
        from google.colab import userdata
        KEY = userdata.get("KOSIS_API_KEY")
    except Exception:
        KEY = os.environ.get("KOSIS_API_KEY")
    parts = []
    for y0 in range(1997, 2026, 3):
        p = dict(method="getList", apiKey=KEY, orgId="101", tblId="DT_1B83A35", itmId="T3", objL1="ALL",
                 prdSe="M", startPrdDe=f"{y0}01", endPrdDe=f"{min(y0 + 2, 2025)}12", format="json", jsonVD="Y")
        url = "https://kosis.kr/openapi/Param/statisticsParameterData.do?" + urllib.parse.urlencode(p)
        rows = json.loads(urllib.request.urlopen(url, timeout=120).read())
        parts.append(pd.DataFrame(rows)[["C1", "C1_NM", "PRD_DE", "DT"]])
        time.sleep(1)
    new = pd.concat(parts).rename(columns={"C1": "code", "C1_NM": "name", "PRD_DE": "ym", "DT": "marriages"})
    os.makedirs("data_kosis", exist_ok=True)
    new.to_csv(DATA, index=False)
    print(len(new), "행 저장 →", DATA)
